# Advanced Problems with Solutions: Python `timeit` and Benchmarking

This notebook contains advanced practice problems on Python's `timeit` module, benchmarking design, namespace handling, setup code, garbage collection effects, and interpreting timing results.

Each problem includes a complete solution and runnable code.

## Setup

Run this cell first.

In [1]:
from timeit import timeit, repeat, Timer
from statistics import mean, median, stdev
import math
import random
import gc
import functools
import operator
from collections import Counter

random.seed(42)

## Problem 1 — Correctly Benchmark Import Styles

Benchmark these two statements:

```python
math.sqrt(2)
sqrt(2)
```

Requirements:

1. Do not include the import cost inside every measured iteration.
2. Run each benchmark for `1_000_000` iterations.
3. Report total time and average nanoseconds per call.
4. Explain why the faster option should not automatically be preferred in production code.

In [2]:
# Solution 1

number = 1_000_000

t_math = timeit(
    stmt="math.sqrt(2)",
    setup="import math",
    number=number
)

t_direct = timeit(
    stmt="sqrt(2)",
    setup="from math import sqrt",
    number=number
)

print(f"math.sqrt total: {t_math:.6f} seconds")
print(f"sqrt total:      {t_direct:.6f} seconds")
print()
print(f"math.sqrt average: {t_math / number * 1e9:.2f} ns/call")
print(f"sqrt average:      {t_direct / number * 1e9:.2f} ns/call")

if t_math < t_direct:
    print("\nmath.sqrt was faster in this run.")
else:
    print("\nsqrt was faster in this run.")

math.sqrt total: 0.116848 seconds
sqrt total:      0.105978 seconds

math.sqrt average: 116.85 ns/call
sqrt average:      105.98 ns/call

sqrt was faster in this run.


### Explanation

The imports are placed in `setup`, so they are executed once before the timed loop begins.

`sqrt(2)` is often slightly faster than `math.sqrt(2)` because Python avoids one attribute lookup. However, this difference is usually tiny compared with real application costs. In production code, readability and clarity usually matter more. `math.sqrt(2)` clearly communicates where `sqrt` comes from.

## Problem 2 — Diagnose a Namespace Failure

The following benchmark fails:

```python
values = [1, 2, 3, 4, 5]
timeit("sum(values)")
```

Tasks:

1. Explain why it fails.
2. Fix it using `globals()`.
3. Fix it using `setup` instead.
4. Explain which fix is better when the object is large.

In [3]:
# Solution 2

values = [1, 2, 3, 4, 5]

# Fix 1: use globals()
t_globals = timeit(
    stmt="sum(values)",
    globals=globals(),
    number=1_000_000
)

# Fix 2: create the object inside setup
t_setup = timeit(
    stmt="sum(values)",
    setup="values = [1, 2, 3, 4, 5]",
    number=1_000_000
)

print(f"Using globals(): {t_globals:.6f} seconds")
print(f"Using setup:     {t_setup:.6f} seconds")

Using globals(): 0.153866 seconds
Using setup:     0.084731 seconds


### Explanation

`timeit` executes the benchmarked statement in a separate namespace. The name `values` is not automatically visible inside that namespace.

`globals=globals()` passes the current global namespace to `timeit`.

`setup` creates `values` inside the namespace used by the benchmark.

For large objects, `globals()` is often more convenient because the object already exists and does not need to be represented as a large setup string. However, using `setup` can make the benchmark more self-contained.

## Problem 3 — Benchmark Local vs Global Lookup

Create two benchmarks that compare reading a variable from:

1. The global namespace.
2. A function local namespace.

Use `timeit` correctly and explain the result.

In [4]:
# Solution 3

x = 10

global_lookup_time = timeit(
    stmt="x + 1",
    globals=globals(),
    number=10_000_000
)

def benchmark_local_lookup():
    x = 10
    return timeit(
        stmt="x + 1",
        globals=locals(),
        number=10_000_000
    )

local_lookup_time = benchmark_local_lookup()

print(f"Global lookup: {global_lookup_time:.6f} seconds")
print(f"Local lookup:  {local_lookup_time:.6f} seconds")

if local_lookup_time < global_lookup_time:
    print("Local lookup was faster in this run.")
else:
    print("Global lookup was faster in this run.")

Global lookup: 0.350594 seconds
Local lookup:  0.220329 seconds
Local lookup was faster in this run.


### Explanation

Local variable access is often faster than global variable access in Python because local variables are stored in a more direct internal structure. Global lookup requires dictionary-style namespace resolution.

Small differences like this are useful for understanding Python internals, but they should rarely drive application design by themselves.

## Problem 4 — Avoid Timing Data Generation by Accident

You want to compare these two ways of counting characters:

```python
Counter(data)
{char: data.count(char) for char in set(data)}
```

The input should be a random list of 10,000 characters.

Requirements:

1. Generate the random data only once.
2. Benchmark both approaches fairly.
3. Explain why including random data generation in the measured statement would be misleading.

In [5]:
# Solution 4

data = random.choices("abcdefghijklmnopqrstuvwxyz", k=10_000)

number = 1_000

counter_time = timeit(
    stmt="Counter(data)",
    globals=globals(),
    number=number
)

count_loop_time = timeit(
    stmt="{char: data.count(char) for char in set(data)}",
    globals=globals(),
    number=number
)

print(f"Counter(data):                      {counter_time:.6f} seconds")
print(f"dict comprehension with count():     {count_loop_time:.6f} seconds")
print(f"Counter speedup:                     {count_loop_time / counter_time:.2f}x")

Counter(data):                      0.527726 seconds
dict comprehension with count():     2.873189 seconds
Counter speedup:                     5.44x


### Explanation

The input data is created once before the benchmark. The timed statements measure only the counting strategies.

If random data generation were placed inside the timed statement, the benchmark would measure both data generation and character counting. That would make the comparison less focused and potentially misleading.

## Problem 5 — Build a Reliable Benchmark Helper

Write a helper function named `benchmark` that:

1. Uses `timeit.repeat`.
2. Accepts `stmt`, `setup`, `globals`, `number`, and `repeat_count`.
3. Returns a dictionary containing:
   - all timings
   - best time
   - median time
   - mean time
   - standard deviation
   - average nanoseconds per operation based on the best time

Then use it to compare `sum(numbers)` and `functools.reduce(operator.add, numbers)`.

In [6]:
# Solution 5

def benchmark(stmt, setup="pass", globals=None, number=1_000_000, repeat_count=7):
    timings = repeat(
        stmt=stmt,
        setup=setup,
        globals=globals,
        number=number,
        repeat=repeat_count
    )
    
    return {
        "timings": timings,
        "best": min(timings),
        "median": median(timings),
        "mean": mean(timings),
        "stdev": stdev(timings) if len(timings) > 1 else 0.0,
        "best_ns_per_op": min(timings) / number * 1e9
    }


numbers = list(range(1_000))

sum_result = benchmark(
    stmt="sum(numbers)",
    globals=globals(),
    number=10_000,
    repeat_count=7
)

reduce_result = benchmark(
    stmt="functools.reduce(operator.add, numbers)",
    globals=globals(),
    number=10_000,
    repeat_count=7
)

print("sum(numbers)")
for key, value in sum_result.items():
    print(f"  {key}: {value}")

print("\nfunctools.reduce(operator.add, numbers)")
for key, value in reduce_result.items():
    print(f"  {key}: {value}")

sum(numbers)
  timings: [0.12214380002114922, 0.09085309994406998, 0.07347599999047816, 0.06841750000603497, 0.07493260002229363, 0.08836819999851286, 0.0748674999922514]
  best: 0.06841750000603497
  median: 0.07493260002229363
  mean: 0.08472267142497003
  stdev: 0.018423395602757107
  best_ns_per_op: 6841.750000603497

functools.reduce(operator.add, numbers)
  timings: [0.3755608999636024, 0.36110830004327, 0.32452000002376735, 0.32779480004683137, 0.3282486000098288, 0.3038116000825539, 0.3111372999846935]
  best: 0.3038116000825539
  median: 0.32779480004683137
  mean: 0.33316878573636394
  stdev: 0.02598983123703283
  best_ns_per_op: 30381.160008255392


### Explanation

`repeat` gives multiple timing samples, which is better than trusting a single timing run.

The best time is often useful because it is the run least affected by temporary system noise. The median and standard deviation help show how stable the benchmark is.

`sum(numbers)` is usually faster than `functools.reduce(operator.add, numbers)` because `sum` is a specialized built-in optimized for numeric summation.

## Problem 6 — Measure the Cost of Function Calls

Compare these three ways of adding two numbers:

1. Inline expression: `a + b`
2. Regular function call: `add(a, b)`
3. Lambda call: `add_lambda(a, b)`

Use `timeit` and explain what is being measured.

In [7]:
# Solution 6

a = 10
b = 20

def add(x, y):
    return x + y

add_lambda = lambda x, y: x + y

number = 10_000_000

inline_time = timeit("a + b", globals=globals(), number=number)
function_time = timeit("add(a, b)", globals=globals(), number=number)
lambda_time = timeit("add_lambda(a, b)", globals=globals(), number=number)

print(f"Inline expression: {inline_time:.6f} seconds")
print(f"Function call:     {function_time:.6f} seconds")
print(f"Lambda call:       {lambda_time:.6f} seconds")
print()
print(f"Function-call overhead vs inline: {(function_time - inline_time) / number * 1e9:.2f} ns/call")
print(f"Lambda-call overhead vs inline:   {(lambda_time - inline_time) / number * 1e9:.2f} ns/call")

Inline expression: 0.362578 seconds
Function call:     0.535223 seconds
Lambda call:       0.539148 seconds

Function-call overhead vs inline: 17.26 ns/call
Lambda-call overhead vs inline:   17.66 ns/call


### Explanation

This benchmark mostly measures function-call overhead. The arithmetic operation itself is extremely cheap, so the difference between the inline expression and the function versions is dominated by the cost of calling a Python function.

This does not mean functions should be avoided. Functions improve readability, reuse, testing, and abstraction.

## Problem 7 — Benchmark Mutation Fairly

You want to compare appending to a list with extending a list by one element:

```python
lst.append(1)
lst.extend([1])
```

Problem: both statements mutate the list.

Design a fair benchmark that avoids one benchmark running on a much larger list than the other.

In [8]:
# Solution 7

# Bad benchmark idea:
# lst = []
# timeit("lst.append(1)", globals=globals(), number=10_000_000)
# This continuously grows the same list.

# Better approach: create a fresh list inside the timed statement for each operation.
# This includes list creation cost, but both statements pay a comparable setup cost.

number = 1_000_000

append_time = timeit(
    stmt="lst = []; lst.append(1)",
    number=number
)

extend_time = timeit(
    stmt="lst = []; lst.extend([1])",
    number=number
)

print(f"Fresh list + append: {append_time:.6f} seconds")
print(f"Fresh list + extend: {extend_time:.6f} seconds")

# Alternative: use setup to create a list and keep number smaller, but mutation still accumulates.
# For mutation benchmarks, always think carefully about state growth.

Fresh list + append: 0.073579 seconds
Fresh list + extend: 0.129144 seconds


### Explanation

Mutating benchmarks are tricky because the operation changes the object being measured.

A benchmark that appends millions of times to the same list measures more than just append cost. It may include resizing behavior and memory effects.

Creating a fresh list in each timed statement keeps the state comparable. The trade-off is that the benchmark also includes list creation cost. For this particular comparison, that is acceptable because both tested statements create the same kind of list.

## Problem 8 — Garbage Collection Effects

`timeit` temporarily disables garbage collection by default.

Create a benchmark where many temporary container objects are created. Compare timing with garbage collection disabled and enabled.

Requirements:

1. Use `gc.enable()` inside setup to force GC to stay enabled during the benchmark.
2. Compare with normal `timeit` behavior.
3. Explain why the result may differ.

In [9]:
# Solution 8

stmt = "[{i: i * i} for i in range(100)]"
number = 100_000

# Default timeit behavior: GC is disabled during timing.
gc_disabled_time = timeit(
    stmt=stmt,
    number=number
)

# Force GC to be enabled during timing.
gc_enabled_time = timeit(
    stmt=stmt,
    setup="import gc; gc.enable()",
    number=number
)

print(f"GC disabled by timeit: {gc_disabled_time:.6f} seconds")
print(f"GC explicitly enabled: {gc_enabled_time:.6f} seconds")

print(f"GC currently enabled after benchmark? {gc.isenabled()}")

GC disabled by timeit: 1.001725 seconds
GC explicitly enabled: 0.895665 seconds
GC currently enabled after benchmark? True


### Explanation

`timeit` disables garbage collection during timing to reduce noise from unpredictable collection pauses.

For code that creates many temporary container objects or reference cycles, GC behavior may matter. If garbage collection is relevant to real-world performance, benchmark both ways or design a higher-level benchmark that includes realistic workload behavior.

## Problem 9 — Detect a Misleading Microbenchmark

A developer writes this benchmark:

```python
timeit("sorted(data)", setup="import random; data = random.sample(range(100000), 10000)")
```

They conclude that sorting is fast enough because the result is good.

Tasks:

1. Identify at least two weaknesses in this benchmark.
2. Improve the benchmark using `repeat`.
3. Test sorting with different input patterns: random, already sorted, and reverse sorted.

In [10]:
# Solution 9

random_data = random.sample(range(100_000), 10_000)
sorted_data = sorted(random_data)
reverse_data = sorted(random_data, reverse=True)

number = 200
repeat_count = 7

cases = {
    "random": "sorted(random_data)",
    "already sorted": "sorted(sorted_data)",
    "reverse sorted": "sorted(reverse_data)"
}

for name, stmt in cases.items():
    timings = repeat(
        stmt=stmt,
        globals=globals(),
        number=number,
        repeat=repeat_count
    )
    print(f"{name} input")
    print(f"  best:   {min(timings):.6f} seconds")
    print(f"  median: {median(timings):.6f} seconds")
    print(f"  all:    {[round(t, 6) for t in timings]}")
    print()

random input
  best:   0.254194 seconds
  median: 0.270134 seconds
  all:    [0.361594, 0.319003, 0.288174, 0.270134, 0.254695, 0.255614, 0.254194]

already sorted input
  best:   0.016443 seconds
  median: 0.020806 seconds
  all:    [0.020806, 0.027032, 0.02621, 0.037454, 0.019872, 0.016443, 0.016521]

reverse sorted input
  best:   0.016617 seconds
  median: 0.018110 seconds
  all:    [0.016998, 0.016857, 0.016617, 0.01811, 0.019594, 0.02663, 0.045223]



### Explanation

Weaknesses in the original benchmark:

1. It tests only one input pattern.
2. It uses a single timing result rather than repeated measurements.
3. It may not represent real application data.
4. It measures `sorted(data)` but does not check whether the application needs a new sorted list or could sort in place.

Python's sorting algorithm, Timsort, performs especially well on data that already has order. Therefore, input pattern matters.

## Problem 10 — Compare `list(map(...))` and List Comprehension

Compare these two approaches:

```python
[math.sqrt(x) for x in numbers]
list(map(math.sqrt, numbers))
```

Requirements:

1. Use a list of 10,000 numbers.
2. Use `repeat`.
3. Report best and median timings.
4. Explain why the result may change depending on the function being called.

In [11]:
# Solution 10

numbers = list(range(1, 10_001))
number = 1_000
repeat_count = 7

list_comp_timings = repeat(
    stmt="[math.sqrt(x) for x in numbers]",
    globals=globals(),
    number=number,
    repeat=repeat_count
)

map_timings = repeat(
    stmt="list(map(math.sqrt, numbers))",
    globals=globals(),
    number=number,
    repeat=repeat_count
)

print("List comprehension")
print(f"  best:   {min(list_comp_timings):.6f} seconds")
print(f"  median: {median(list_comp_timings):.6f} seconds")

print("\nlist(map(...))")
print(f"  best:   {min(map_timings):.6f} seconds")
print(f"  median: {median(map_timings):.6f} seconds")

List comprehension
  best:   0.881085 seconds
  median: 0.907878 seconds

list(map(...))
  best:   0.778911 seconds
  median: 0.854042 seconds


### Explanation

`map` can perform well when applying an existing function implemented in C, such as `math.sqrt`.

List comprehensions often perform very well for inline Python expressions because they avoid some function-call overhead and are optimized by Python.

The best choice depends on the operation, readability, and whether a named function already exists.

## Problem 11 — Benchmark a Decorator Without Measuring Decoration Time

Suppose you have this decorator:

```python
def trace_calls(fn):
    def inner(*args, **kwargs):
        return fn(*args, **kwargs)
    return inner
```

Benchmark a decorated function against an undecorated function.

Requirements:

1. Do not include the cost of creating the decorated function in the timed loop.
2. Measure only call overhead.
3. Explain the result.

In [12]:
# Solution 11

def trace_calls(fn):
    def inner(*args, **kwargs):
        return fn(*args, **kwargs)
    return inner

def plain_add(x, y):
    return x + y

decorated_add = trace_calls(plain_add)

number = 5_000_000

plain_time = timeit(
    stmt="plain_add(10, 20)",
    globals=globals(),
    number=number
)

decorated_time = timeit(
    stmt="decorated_add(10, 20)",
    globals=globals(),
    number=number
)

print(f"Plain function:     {plain_time:.6f} seconds")
print(f"Decorated function: {decorated_time:.6f} seconds")
print(f"Extra cost:         {(decorated_time - plain_time) / number * 1e9:.2f} ns/call")

Plain function:     0.342428 seconds
Decorated function: 0.887571 seconds
Extra cost:         109.03 ns/call


### Explanation

The decorated function is created before timing starts. Therefore, the benchmark measures only the extra call layer introduced by the decorator.

The decorated function usually takes longer because calling it requires one additional Python function call before reaching the original function.

## Problem 12 — Use `Timer` for Reusable Benchmark Objects

Create reusable `Timer` objects for these statements:

```python
text.lower()
text.casefold()
```

Use a string containing mixed-case Unicode text.

Run each timer several times and compare the results.

In [13]:
# Solution 12

text = "Straße İSTANBUL Python Μάθημα " * 1_000

lower_timer = Timer(
    stmt="text.lower()",
    globals=globals()
)

casefold_timer = Timer(
    stmt="text.casefold()",
    globals=globals()
)

number = 10_000
repeat_count = 5

lower_timings = lower_timer.repeat(repeat=repeat_count, number=number)
casefold_timings = casefold_timer.repeat(repeat=repeat_count, number=number)

print("text.lower()")
print(f"  best:   {min(lower_timings):.6f} seconds")
print(f"  median: {median(lower_timings):.6f} seconds")

print("\ntext.casefold()")
print(f"  best:   {min(casefold_timings):.6f} seconds")
print(f"  median: {median(casefold_timings):.6f} seconds")

text.lower()
  best:   1.928575 seconds
  median: 1.950791 seconds

text.casefold()
  best:   2.387662 seconds
  median: 2.451881 seconds


### Explanation

`Timer` is useful when you want to define a benchmark once and execute it repeatedly.

`casefold()` is designed for more aggressive Unicode normalization for caseless matching. It can do more work than `lower()`, so it may be slower. The correct choice depends on correctness requirements, not just speed.

## Problem 13 — Choose the Right `number`

Write a small function that automatically increases `number` until the total benchmark time is at least `0.2` seconds.

Use it to benchmark:

```python
x * x
sum(range(1000))
```

Explain why this is useful.

In [14]:
# Solution 13

def calibrate_number(stmt, setup="pass", globals=None, target_seconds=0.2, start=1):
    number = start
    while True:
        elapsed = timeit(stmt=stmt, setup=setup, globals=globals, number=number)
        if elapsed >= target_seconds:
            return number, elapsed
        number *= 10


x = 123

for stmt in ["x * x", "sum(range(1000))"]:
    number, elapsed = calibrate_number(stmt, globals=globals())
    print(f"Statement: {stmt}")
    print(f"  chosen number: {number}")
    print(f"  elapsed:       {elapsed:.6f} seconds")
    print(f"  ns/op:         {elapsed / number * 1e9:.2f}")
    print()

Statement: x * x
  chosen number: 10000000
  elapsed:       0.364042 seconds
  ns/op:         36.40

Statement: sum(range(1000))
  chosen number: 100000
  elapsed:       1.411146 seconds
  ns/op:         14111.46



### Explanation

Very fast operations need many iterations to produce a measurable total time. Slow operations need fewer iterations.

Calibrating `number` helps avoid timings that are dominated by timer resolution, system noise, or loop overhead.

## Problem 14 — Benchmark Algorithmic Complexity

Compare membership testing in a list and in a set for increasing input sizes.

Sizes:

```python
100, 1_000, 10_000, 100_000
```

For each size, test whether `-1` is present.

Explain the observed trend.

In [15]:
# Solution 14

sizes = [100, 1_000, 10_000, 100_000]
number = 10_000

for size in sizes:
    data_list = list(range(size))
    data_set = set(data_list)
    target = -1

    list_time = timeit(
        stmt="target in data_list",
        globals=locals(),
        number=number
    )

    set_time = timeit(
        stmt="target in data_set",
        globals=locals(),
        number=number
    )

    print(f"Size: {size}")
    print(f"  list membership: {list_time:.6f} seconds")
    print(f"  set membership:  {set_time:.6f} seconds")
    print(f"  ratio:           {list_time / set_time:.2f}x")
    print()

Size: 100
  list membership: 0.012460 seconds
  set membership:  0.000308 seconds
  ratio:           40.40x

Size: 1000
  list membership: 0.110087 seconds
  set membership:  0.000230 seconds
  ratio:           477.81x

Size: 10000
  list membership: 0.934722 seconds
  set membership:  0.000449 seconds
  ratio:           2083.18x

Size: 100000
  list membership: 8.480443 seconds
  set membership:  0.000228 seconds
  ratio:           37211.25x



### Explanation

A failed membership test in a list must scan the list, so it grows roughly linearly with input size.

A set uses hashing, so average membership testing is approximately constant time.

This is an example where benchmarking reveals algorithmic complexity, not just tiny implementation details.

## Problem 15 — Final Challenge: Design a Benchmark Report

Create a benchmark report comparing three ways to flatten a list of lists:

1. Nested list comprehension
2. `sum(matrix, [])`
3. `itertools.chain.from_iterable(matrix)`

Requirements:

1. Use a matrix with 1,000 rows and 100 columns.
2. Use `repeat`.
3. Report best, median, and nanoseconds per full flattening operation.
4. Explain which result is best and why `sum(matrix, [])` is usually problematic.

In [16]:
# Solution 15

import itertools

matrix = [list(range(100)) for _ in range(1_000)]

benchmarks = {
    "nested list comprehension": "[item for row in matrix for item in row]",
    "sum(matrix, [])": "sum(matrix, [])",
    "itertools.chain.from_iterable": "list(itertools.chain.from_iterable(matrix))"
}

number = 100
repeat_count = 7

report = []

for name, stmt in benchmarks.items():
    timings = repeat(
        stmt=stmt,
        globals=globals(),
        number=number,
        repeat=repeat_count
    )
    best = min(timings)
    med = median(timings)
    report.append((name, best, med, best / number * 1e9))

report.sort(key=lambda row: row[1])

print(f"{'method':35} {'best seconds':>15} {'median seconds':>15} {'best ns/op':>15}")
print("-" * 85)
for name, best, med, ns_per_op in report:
    print(f"{name:35} {best:15.6f} {med:15.6f} {ns_per_op:15.2f}")

method                                 best seconds  median seconds      best ns/op
-------------------------------------------------------------------------------------
itertools.chain.from_iterable              0.070978        0.075551       709783.00
nested list comprehension                  0.128967        0.158132      1289674.00
sum(matrix, [])                            5.992622        6.174203     59926221.00


### Explanation

`itertools.chain.from_iterable(matrix)` and the nested list comprehension are usually strong choices.

`sum(matrix, [])` is usually problematic because list addition repeatedly creates new intermediate lists. As the accumulated list grows, each new addition copies more data. This can lead to much worse scaling behavior.

The best benchmark result may vary by Python version and machine, but `sum(matrix, [])` is typically the method to avoid.

# Benchmarking Best Practices Summary

1. Put imports and one-time setup in `setup`, not inside the measured statement.
2. Use `globals=globals()` or `globals=locals()` when the timed code needs existing names.
3. Remember that `timeit` returns total time, not average time.
4. Use `repeat` instead of trusting one run.
5. Report best and median timings.
6. Be careful with benchmarks that mutate state.
7. Do not accidentally benchmark data generation unless that is intentional.
8. Understand that `timeit` disables garbage collection by default.
9. Prefer clear code unless benchmarking proves performance matters.
10. Benchmark realistic workloads, not only tiny isolated operations.